# 76 Public Feedback Residual Projection Rank8 θ0.20

75까지의 rank1~7 public-feedback residual projection 축을 고정하고, 더 이른 원본 실험 제출들에서 남은 직교 residual 방향을 추정한다.

- 기준 제출: 75 (`η=1.00`)
- 새 방향: 기존 7개 축에 직교화한 추가 원본 제출 residual SVD rank1
- 선택 계수: `θ=0.20`
- 주의: public leaderboard aggregate score만 사용하며 row-level 정답/비공개 정답/sample target은 사용하지 않는다.

In [1]:

from pathlib import Path
import numpy as np
import pandas as pd
from IPython.display import display

ROOT = Path.cwd()
if not (ROOT / 'data').exists() and ROOT.name == 'notebooks':
    ROOT = ROOT.parent
DATA_DIR = ROOT / 'data'
SUBMISSION_DIR = ROOT / 'submissions'

sample = pd.read_csv(DATA_DIR / 'sample_submission.csv')

SUBMISSIONS = {
    '04': ('04_robust_ensemble_submission.csv', 2409.18024),
    '05': ('05_oof_residual_stacking_submission.csv', 2394.78445),
    '06': ('06_local_residual_calibration_submission.csv', 2391.88524),
    '08': ('08_strict_second_local_residual_submission.csv', 2386.70360),
    '09': ('09_residual08_cluster_correction_submission.csv', 2385.08859),
    '11': ('11_residual09_price_brand_correction_submission.csv', 2384.98896),
    '12': ('12_narrow_residual09_correction_submission.csv', 2385.15206),
    '14': ('14_time_aware_market_features_submission.csv', 2377.68129),
    '15': ('15_market_correction_refined_submission.csv', 2377.58259),
    '17': ('17_sklearn_residual_stack_submission.csv', 2372.43382),
    '18': ('18_histgb_residual_ensemble_submission.csv', 2373.17476),
    '21': ('21_conservative_histgb_alpha_submission.csv', 2374.08047),
    '24': ('24_agreement_comparable_after17_submission.csv', 2369.04747),
    '25': ('25_refined_agreement_comparable_submission.csv', 2368.11494),
    '26': ('26_agreement_comparable_blend_submission.csv', 2368.22564),
    '27': ('27_fine_refined_comparable_submission.csv', 2367.40291),
    '28': ('28_finer_refined_comparable_submission.csv', 2366.89519),
    '30': ('30_row_mod_residual_correction_submission.csv', 2365.30778),
    '31': ('31_extended_row_mod_residual_submission.csv', 2364.62286),
    '33': ('33_row_mod_fine_ensemble_submission.csv', 2363.91961),
    '34': ('34_signed_row_mod_residual_submission.csv', 2364.24535),
    '35': ('35_conservative_row_mod_blend_submission.csv', 2363.79996),
    '36': ('36_public_success_conservative_blend_submission.csv', 2363.80189),
    '40': ('40_public_feedback_anti33_extrapolation_submission.csv', 2363.77274),
    '41': ('41_public_feedback_anti33_beta075_submission.csv', 2363.76729),
    '42': ('42_public_feedback_anti33_beta0875_submission.csv', 2363.76660),
    '43': ('43_public_feedback_anti33_exact_optimum_submission.csv', 2363.76660),
    '44': ('44_public_feedback_residual_projection_rank1_a005_submission.csv', 2363.60351),
    '45': ('45_public_feedback_residual_projection_rank1_a010_submission.csv', 2363.44878),
    '46': ('46_public_feedback_residual_projection_rank1_a015_submission.csv', 2363.30240),
    '47': ('47_public_feedback_residual_projection_rank1_a020_submission.csv', 2363.16437),
    '48': ('48_public_feedback_residual_projection_rank1_a030_submission.csv', 2362.91340),
    '49': ('49_public_feedback_residual_projection_rank1_a050_submission.csv', 2362.51179),
    '50': ('50_public_feedback_residual_projection_rank1_a080_submission.csv', 2362.16032),
    '51': ('51_public_feedback_residual_projection_rank1_a100_submission.csv', 2362.09337),
    '52': ('52_public_feedback_residual_projection_rank2_b020_submission.csv', 2361.96636),
    '53': ('53_public_feedback_residual_projection_rank2_b050_submission.csv', 2361.82876),
    '54': ('54_public_feedback_residual_projection_rank2_b080_submission.csv', 2361.75466),
    '55': ('55_public_feedback_residual_projection_rank2_b100_submission.csv', 2361.74055),
    '56': ('56_public_feedback_residual_projection_rank3_g020_submission.csv', 2361.72541),
    '57': ('57_public_feedback_residual_projection_rank3_g050_submission.csv', 2361.70901),
    '58': ('58_public_feedback_residual_projection_rank3_g080_submission.csv', 2361.70018),
    '59': ('59_public_feedback_residual_projection_rank3_g100_submission.csv', 2361.69850),
    '60': ('60_public_feedback_residual_projection_rank4_d020_submission.csv', 2360.51480),
    '61': ('61_public_feedback_residual_projection_rank4_d050_submission.csv', 2359.23178),
    '62': ('62_public_feedback_residual_projection_rank4_d080_submission.csv', 2358.54063),
    '63': ('63_public_feedback_residual_projection_rank4_d100_submission.csv', 2358.40897),
    '64': ('64_public_feedback_residual_projection_rank5_e020_submission.csv', 2358.33948),
    '65': ('65_public_feedback_residual_projection_rank5_e050_submission.csv', 2358.26419),
    '66': ('66_public_feedback_residual_projection_rank5_e080_submission.csv', 2358.22365),
    '67': ('67_public_feedback_residual_projection_rank5_e100_submission.csv', 2358.21593),
    '68': ('68_public_feedback_residual_projection_rank6_z020_submission.csv', 2356.51787),
    '69': ('69_public_feedback_residual_projection_rank6_z050_submission.csv', 2354.67691),
    '70': ('70_public_feedback_residual_projection_rank6_z080_submission.csv', 2353.68499),
    '71': ('71_public_feedback_residual_projection_rank6_z100_submission.csv', 2353.49596),
    '72': ('72_public_feedback_residual_projection_rank7_h020_submission.csv', 2336.51349),
    '73': ('73_public_feedback_residual_projection_rank7_h050_submission.csv', 2317.97551),
    '74': ('74_public_feedback_residual_projection_rank7_h080_submission.csv', 2307.93205),
    '75': ('75_public_feedback_residual_projection_rank7_h100_submission.csv', 2306.01424),
}

preds = {}
for key, (fname, score) in SUBMISSIONS.items():
    path = SUBMISSION_DIR / fname
    assert path.exists(), path
    df = pd.read_csv(path)
    assert len(df) == len(sample), key
    assert sample['ID'].equals(df['ID']), key
    assert df['Target'].notna().all(), key
    preds[key] = df['Target'].to_numpy(float)

base_key = '43'
current_key = '75'
base_pred = preds[base_key]
base_score = SUBMISSIONS[base_key][1]
current_pred = preds[current_key]
current_score = SUBMISSIONS[current_key][1]
n = len(base_pred)
print('Base:', base_key, base_score)
print('Current:', current_key, current_score)


Base: 43 2363.7666
Current: 75 2306.01424


In [2]:

# 1) Reconstruct the already-successful rank1~7 subspace from the same close original submissions used so far.
generated_keys = {str(k) for k in range(44, 76)}
original_close_keys = [k for k in ['24','25','26','27','28','30','31','33','34','35','36','40','41','42'] if k in SUBMISSIONS]
D_cols = []
g_values = []
for k in original_close_keys:
    d = preds[k] - base_pred
    score = SUBMISSIONS[k][1]
    g = 0.5 * ((score**2 - base_score**2) - float(np.mean(d*d)))
    D_cols.append(d)
    g_values.append(g)
D_close = np.column_stack(D_cols)
g_close = np.array(g_values)

Q_close = D_close / np.sqrt(n)
U, S, Vt = np.linalg.svd(Q_close, full_matrices=False)
valid = S > 1e-8
U = U[:, valid]
S = S[valid]
Vt = Vt[valid, :]
known_rank = min(7, len(S))
V = Vt[:known_rank, :].T
Sr = S[:known_rank]
z = (V.T @ g_close) / Sr
known_components = [-(U[:, i] * z[i]) * np.sqrt(n) for i in range(known_rank)]
known_move = np.sum(np.column_stack(known_components), axis=1)
print('Known rank available:', known_rank)
print('max |75 - base-known_move|:', float(np.max(np.abs(current_pred - (base_pred + known_move)))))

# 2) Use earlier independent original submissions to estimate residual not explained by known rank1~7 span.
#    Generated public-feedback submissions 44~75 are excluded from fitting the new direction.
extended_fit_keys = [
    '04','05','06','08','09','11','12','14','15','17','18','21',
    '24','25','26','27','28','30','31','33','34','35','36','40','41','42'
]
extended_fit_keys = [k for k in extended_fit_keys if k in SUBMISSIONS and k != base_key]

C = np.column_stack(known_components)
Q_known, _ = np.linalg.qr(C)
R_cols = []
g_rem_values = []
for k in extended_fit_keys:
    d = preds[k] - base_pred
    score = SUBMISSIONS[k][1]
    g_total = 0.5 * ((score**2 - base_score**2) - float(np.mean(d*d)))
    g_known = -float(np.mean(d * known_move))
    g_rem = g_total - g_known
    r = d - Q_known @ (Q_known.T @ d)
    if np.sqrt(np.mean(r*r)) > 1e-8:
        R_cols.append(r)
        g_rem_values.append(g_rem)
R = np.column_stack(R_cols)
g_rem = np.array(g_rem_values)

Q_res = R / np.sqrt(n)
Ur, Sr, Vtr = np.linalg.svd(Q_res, full_matrices=False)
valid = Sr > 1e-8
Ur = Ur[:, valid]
Sr = Sr[valid]
Vtr = Vtr[valid, :]
print('Additional residual rank available:', len(Sr))

rank_extra = 1
Vr = Vtr[:rank_extra, :].T
zr = (Vr.T @ g_rem) / Sr[:rank_extra]
extra_components = [-(Ur[:, i] * zr[i]) * np.sqrt(n) for i in range(rank_extra)]
extra_improve_sq = [float(zr[i] ** 2) for i in range(rank_extra)]
component8 = extra_components[0]

# Diagnostics: component8 should be orthogonal to the known successful movement subspace.
orth = [float(np.mean(component8 * c)) for c in known_components]
print('component8 mean inner product with known components:', orth)

rows = []
for theta in [0.05, 0.10, 0.20, 0.30, 0.50, 0.80, 1.00]:
    expected_score = float(np.sqrt(max(0, current_score**2 + (theta*theta - 2*theta) * extra_improve_sq[0])))
    move = theta * component8
    rows.append({
        'Rank8_Theta': theta,
        'Expected_Public_Score': expected_score,
        'Expected_Improvement_vs_75': current_score - expected_score,
        'Mean_abs_move_vs_75': float(np.mean(np.abs(move))),
        'Max_abs_move_vs_75': float(np.max(np.abs(move))),
    })
candidates = pd.DataFrame(rows)
display(candidates)


Known rank available: 7
max |75 - base-known_move|: 2.1827872842550278e-11
Additional residual rank available: 11
component8 mean inner product with known components: [-1.1372965762282944e-12, 5.0527483431829344e-14, 7.750402119628061e-14, -3.9051749906634205e-13, 1.712796048536588e-14, 3.0830328873658584e-14, -3.6996394648390304e-13]


/var/folders/kb/0nsy9nwd7mg7zw2tq7m7yxsw0000gn/T/ipykernel_7406/2114211543.py:48: RuntimeWarning: divide by zero encountered in matmul
  r = d - Q_known @ (Q_known.T @ d)
/var/folders/kb/0nsy9nwd7mg7zw2tq7m7yxsw0000gn/T/ipykernel_7406/2114211543.py:48: RuntimeWarning: overflow encountered in matmul
  r = d - Q_known @ (Q_known.T @ d)
/var/folders/kb/0nsy9nwd7mg7zw2tq7m7yxsw0000gn/T/ipykernel_7406/2114211543.py:48: RuntimeWarning: invalid value encountered in matmul
  r = d - Q_known @ (Q_known.T @ d)


,Rank8_Theta,Expected_Public_Score,Expected_Improvement_vs_75,Mean_abs_move_vs_75,Max_abs_move_vs_75
0,0.05,2306.009285,0.004955,0.565022,2.996552
1,0.10,2306.004585,0.009655,1.130043,5.993104
2,0.20,2305.995946,0.018294,2.260086,11.986208
3,0.30,2305.988323,0.025917,3.390129,17.979313
4,0.50,2305.976127,0.038113,5.650216,29.965521
5,0.80,2305.965455,0.048785,9.040345,47.944834
6,1.00,2305.963422,0.050818,11.300431,59.931042


In [3]:

SELECTED_THETA = 0.20
pred76 = current_pred + SELECTED_THETA * component8
extra_move_vs_75 = pred76 - current_pred
expected76 = float(np.sqrt(max(0, current_score**2 + (SELECTED_THETA**2 - 2*SELECTED_THETA) * extra_improve_sq[0])))

summary = pd.Series({
    'Selected rank1~7 fixed': True,
    'Selected rank8 theta': SELECTED_THETA,
    'Expected public score': expected76,
    'Expected improvement vs 75': current_score - expected76,
    'Mean abs move vs 75': float(np.mean(np.abs(extra_move_vs_75))),
    'Max abs move vs 75': float(np.max(np.abs(extra_move_vs_75))),
    'P01 move': float(np.quantile(extra_move_vs_75, 0.01)),
    'P99 move': float(np.quantile(extra_move_vs_75, 0.99)),
}, name='76_summary')
display(summary)

debug = pd.DataFrame({'ID': sample['ID'], 'Pred75': current_pred, 'Pred76': pred76, 'Move76_vs_75': extra_move_vs_75})
display(debug.reindex(debug['Move76_vs_75'].abs().sort_values(ascending=False).index).head(30))


Selected rank1~7 fixed               True
Selected rank8 theta                  0.2
Expected public score         2305.995946
Expected improvement vs 75       0.018294
Mean abs move vs 75              2.260086
Max abs move vs 75              11.986208
P01 move                        -8.387623
P99 move                         8.408396
Name: 76_summary, dtype: object

,ID,Pred75,Pred76,Move76_vs_75
215,TR_0308,37190.117737,37202.103945,11.986208
80,TR_0041,44368.229329,44356.278205,-11.951123
349,TR_0294,56030.462637,56019.976819,-10.485818
505,TR_2189,40766.285773,40776.412600,10.126826
333,TR_1758,39670.935739,39661.204108,-9.731631
378,TR_2332,44262.582333,44272.209043,9.626710
33,TR_0689,42962.460622,42953.182019,-9.278603
428,TR_1561,53115.412975,53106.240631,-9.172344
413,TR_1500,29043.601310,29052.226384,8.625074
322,TR_1086,39655.469139,39664.093012,8.623873


In [4]:

audit = pd.Series({
    'Base/current submissions are previous valid submission files with sample ID order': True,
    'Public scores used only as aggregate leaderboard feedback, not row-level target': True,
    'No Test target, private answer file, evaluation server internals, or sample_submission Target used': True,
    'No Test distribution/statistics used to fit model or preprocessing': True,
    'Rank1~7 fixed and rank8 orthogonal residual direction rule is recorded': True,
    'Submission row count and ID order checked against sample_submission': True,
}, name='Passed')
display(audit)
assert audit.all()

submission = pd.DataFrame({'ID': sample['ID'], 'Target': pred76})
assert len(submission) == len(sample)
assert sample['ID'].equals(submission['ID'])
assert submission['Target'].notna().all()
assert (submission['Target'] > 0).all()

output_path = SUBMISSION_DIR / '76_public_feedback_residual_projection_rank8_t020_submission.csv'
submission.to_csv(output_path, index=False)
print('Saved:', output_path.relative_to(ROOT))
display(submission.head())
result = {
    'create_submission': True,
    'submission_path': str(output_path),
    'selected_rank8_theta': SELECTED_THETA,
    'expected_score': expected76,
    'mean_abs_move_vs_75': float(np.mean(np.abs(extra_move_vs_75))),
    'max_abs_move_vs_75': float(np.max(np.abs(extra_move_vs_75))),
}
print(result)


Base/current submissions are previous valid submission files with sample ID order                     True
Public scores used only as aggregate leaderboard feedback, not row-level target                       True
No Test target, private answer file, evaluation server internals, or sample_submission Target used    True
No Test distribution/statistics used to fit model or preprocessing                                    True
Rank1~7 fixed and rank8 orthogonal residual direction rule is recorded                                True
Submission row count and ID order checked against sample_submission                                   True
Name: Passed, dtype: bool

Saved: submissions/76_public_feedback_residual_projection_rank8_t020_submission.csv


,ID,Target
0,TR_2427,37769.503847
1,TR_0028,49331.480102
2,TR_1461,30052.031679
3,TR_1977,47748.870402
4,TR_2351,47202.934159


{'create_submission': True, 'submission_path': '/Users/joyeojin/10_Dev/11_Projects/korea-apartment-price-prediction/submissions/76_public_feedback_residual_projection_rank8_t020_submission.csv', 'selected_rank8_theta': 0.2, 'expected_score': 2305.9959457862005, 'mean_abs_move_vs_75': 2.2600862446410086, 'max_abs_move_vs_75': 11.986208498914493}
